In [ ]:
# bootstrap: Colab clone + local import of `agentlab` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "07-application-agent-framework/agent-fundamentals/gcp-agent-platform-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "agentlab").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 00 · Setup and the fake model

Everything in this lab runs **offline**. A scripted `FakeLLM` takes the place of Gemini.
Thus you can learn the mechanics that a design review really examines, and you need no API key.
These mechanics are tool contracts, loops, state, identity, evaluation and cost.
When you have a key, `agentlab.llm.gemini.GeminiLLM` is a drop-in replacement (see `docs/GEMINI_ADAPTER.md`).

**Concept map:** see [docs/PRIMER_MAP.md](../docs/PRIMER_MAP.md).

In this notebook, you will:
1. Operate a `FakeLLM` in three ways: a scripted queue, a policy function and a `KeywordPlanner`.
2. See how the model reports token usage and *context caching*.
3. Write your own planner policy. It has the same shape as the output of a real tool-calling model.

> **How to use this notebook.** Run the cells top to bottom. Cells marked **Exercise** contain
> `# YOUR CODE HERE`. Replace it with your implementation, then run the **Check** cell below it.
> A check that passes prints nothing or ✅; a failing assertion tells you what is still wrong.
> The fully solved version is in `solutions/` — try the exercise first, then compare.


In [ ]:
from agentlab.llm import FakeLLM, KeywordPlanner, Rule, call, calls, scripted, text, count_tokens

## 1. A scripted model

`scripted(...)` returns one entry for each model call, in order. A string becomes a plain text answer.
`call(...)` and `calls(...)` become tool-call turns. These are exactly the two shapes that a function-calling model produces.

In [ ]:
llm = scripted(
    call("get_balance", account_id="acc-1"),          # 1st call: the model asks for a tool
    "Your balance is SGD 1,234.50.",                   # 2nd call: the model answers
)
r1 = await llm.generate([{"role": "user", "content": "What's my balance?"}], tools=[{"name": "get_balance"}])
print("tool calls:", [(tc.name, tc.args) for tc in r1.tool_calls], "| finish:", r1.finish_reason)
r2 = await llm.generate([
    {"role": "user", "content": "What's my balance?"},
    r1.as_message(),
    {"role": "tool", "name": "get_balance", "tool_call_id": r1.tool_calls[0].id, "content": '{"ok":true,"data":{"balance":1234.5}}'},
])
print("text:", r2.text, "| usage:", r2.usage)

Every response has `usage` (the input, output and cached tokens) and a simulated `latency_ms`.
The token count is an estimate (≈ 4 characters per token).
The estimate is sufficient to make the budgets and the cost calculations behave as in production.

In [ ]:
print("tokens in 'Hello, agentic world':", count_tokens("Hello, agentic world"))
print("latency of the last call (ms):", round(r2.latency_ms, 1))

## 2. Context caching, simulated

A real model bills the *stable prefix* of a prompt at a large discount when the prefix repeats.
`FakeLLM` copies this behaviour. When calls start with the same messages, it reports those tokens as `cached_tokens`.
This is why the prompt **layout** (the stable material first) is a control for cost. Notebooks 04 and 12 measure it.

In [ ]:
policy_text = "Refund policy: " + "items may be returned within 30 days with proof of purchase. " * 40
cache_llm = FakeLLM(responses=["ok", "ok", "ok"])
system = {"role": "system", "content": policy_text}
for turn in ("Can I return shoes?", "What about electronics?", "And gift cards?"):
    r = await cache_llm.generate([system, {"role": "user", "content": turn}])
    print(f"{turn:28s} input={r.usage.input_tokens:5d}  cached={r.usage.cached_tokens:5d}  "
          f"share={r.usage.cached_tokens / r.usage.input_tokens:.0%}")

### Exercise 2.1 — break the cache, then explain it

Change *one thing* in how you build the messages, so that the second and third turns cache **no** tokens.
Do not change the policy text. (Hint: what occurs if the volatile part comes *before* the stable part?)

Then write one sentence in `explanation`. Say why that order is high-cost in production.

In [ ]:
def build_uncacheable_messages(turn: str) -> list[dict]:
    """Return messages for one turn such that the prefix cache never hits."""
    # YOUR CODE HERE
    raise NotImplementedError("replace this with your implementation")

# YOUR CODE HERE
raise NotImplementedError("replace this with your implementation")

In [ ]:
probe = FakeLLM(responses=["ok"] * 3)
cached = []
for t in ("Can I return shoes?", "What about electronics?", "And gift cards?"):
    r = await probe.generate(build_uncacheable_messages(t))
    cached.append(r.usage.cached_tokens)
assert cached == [0, 0, 0], f"expected no cache hits, got {cached}"
assert isinstance(explanation, str) and len(explanation) > 40, "write a real sentence"
print("✅ cache broken as intended:", cached)

## 3. A policy-driven model

A **policy** is any function `(messages, tools) -> ModelResponse | str | list[ToolCall]`.
`KeywordPlanner` is a ready-made policy, with two behaviours:
- When the latest user message contains keywords, it returns tool calls.
- When tool results are present, it returns a final answer from a template.

It has no intelligence, and this is intentional. The lab is about the *harness* around the model.
Production systems succeed or fail in the harness.

In [ ]:
planner = KeywordPlanner(
    rules=[
        Rule(r"balance", "get_balance", lambda t: {"account_id": "acc-1"}),
        Rule(r"block .*card", "block_card", lambda t: {"card_id": "card-9"}),
    ],
    answer_template="Done. {results}",
)
p_llm = FakeLLM(policy=planner)
tools = [{"name": "get_balance"}, {"name": "block_card"}]
r = await p_llm.generate([{"role": "user", "content": "Block my card and show my balance"}], tools=tools)
print("two rules matched → two parallel tool calls:", [tc.name for tc in r.tool_calls])

### Exercise 3.1 — extract arguments with a regex

Write a `Rule` whose `args` function gets the order id from messages such as
*"where is order ORD-10442?"* and *"status of ORD-7?"*. The function returns `{"order_id": "ORD-10442"}`.
If no id is present, return `{"order_id": None}`.
Then the schema validation of the tool produces a useful error (Notebook 01).

In [ ]:
import re

def order_args(user_text: str) -> dict:
    # YOUR CODE HERE
    raise NotImplementedError("replace this with your implementation")

order_rule = Rule(r"order", "get_order_status", order_args)

In [ ]:
assert order_args("where is order ORD-10442?") == {"order_id": "ORD-10442"}
assert order_args("status of ord-7?") == {"order_id": "ORD-7"}
assert order_args("where is my order?") == {"order_id": None}
o_llm = FakeLLM(policy=KeywordPlanner([order_rule]))
resp = await o_llm.generate([{"role": "user", "content": "where is order ORD-10442?"}], tools=[{"name": "get_order_status"}])
assert resp.tool_calls[0].args == {"order_id": "ORD-10442"}
print("✅ order rule works")

### Exercise 3.2 — write a policy from scratch

Write `triage_policy(messages, tools)` with this behaviour:

* If the messages after the last user turn contain a **tool result**, return the text
  `"Resolved: <content of the last tool result>"`.
* If not, and the last user message contains `"refund"`, return one tool call `issue_refund(order_id="ORD-1", amount=20.0)`.
* If not, return the text `"How can I help?"`.

Use the helpers `text(...)` and `call(...)`. Every function-calling model produces this same decision shape at each step.

In [ ]:
def triage_policy(messages, tools):
    # YOUR CODE HERE
    raise NotImplementedError("replace this with your implementation")

In [ ]:
t_llm = FakeLLM(policy=triage_policy)
a = await t_llm.generate([{"role": "user", "content": "I want a refund"}])
assert a.tool_calls and a.tool_calls[0].name == "issue_refund" and a.tool_calls[0].args["amount"] == 20.0
b = await t_llm.generate([{"role": "user", "content": "I want a refund"}, a.as_message(),
                          {"role": "tool", "name": "issue_refund", "content": '{"ok":true}'}])
assert b.text == 'Resolved: {"ok":true}', b.text
c = await t_llm.generate([{"role": "user", "content": "hello"}])
assert c.text == "How can I help?"
print("✅ triage policy behaves like a tool-calling model")

## 4. What the model recorded

`FakeLLM.calls` keeps every request that it received. Exercises in later notebooks make assertions on it.
Two examples: "did the agent send the tool result back?" and "how many model calls did this turn cost?".

In [ ]:
print("calls made to t_llm:", t_llm.call_count)
print("tools offered on the last call:", t_llm.calls[-1]["tools"])

## The one-minute version

When a person asks *"how does the agent decide what to do?"*, the answer is the policy shape that you wrote.
The model returns text or structured tool calls against a schema.
Then **the runtime** validates the calls, executes them, appends the results and enforces the budgets.
The harness sets the limits of the intelligence. Notebook 01 is about the harness.